In [1]:
import numpy as np

In [2]:
# Set seed for reproducible initial weights
np.random.seed(42)

# The sequence and it's predictions : 
corpus = ["a","b","c"]
targets = ["b","c"]

# These are raw values which cannot be passed so encode them into numeric value 
# For extraction of such words we have a systematic approach of extracting all the unique values from the text at once to build vocabulary

vocabulary = sorted(list(set(" ".join(corpus).split())))

vocab_size = len(vocabulary)

# Create a set of dictionaries to map the tokens to unique id
word_to_idx = {word : i for i,word in enumerate(vocabulary)}
idx_to_word = {idx : word for idx,word in enumerate(vocabulary)}

# Next step is to conver our text in Numeric form for which we will just onehotencode them into dimension i .
# Note :  i is the no of input values and also the max words in corpus 

# X_words will be ['a', 'b'], y_words will be ['b', 'c']
X_words = corpus[:-1]
y_words = corpus[1:]

# Convert your word sequences into integer index lists using your dictionary
X_indices = [word_to_idx[word] for word in corpus[:-1]]  # [0, 1]
y_indices = [word_to_idx[word] for word in corpus[1:]]   # [1, 2]

# The Trick: Use the indices to grab rows from a 3x3 Identity Matrix
X_train = np.eye(vocab_size)[X_indices]
y_train = np.eye(vocab_size)[y_indices]

In [3]:
X_train

array([[1., 0., 0.],
       [0., 1., 0.]])

In [4]:
y_train

array([[0., 1., 0.],
       [0., 0., 1.]])

In [11]:
# Now set the initial values of each parameter -- used to iterate later.
input_dim = 3
hidden_dim = 2
output_dim = 3

# 1. Weights from Input Layer to Hidden Layer (Shape: 3 inputs x 2 hidden nodes)
W_xh = np.random.randn(input_dim, hidden_dim) * 0.01 # Initialize with a random small number
b_h = np.zeros((1, hidden_dim))

# 2. Weights from Hidden Layer to Hidden State (Shape: 2 hidden nodes x 3 outputs)
W_hh = np.random.randn(hidden_dim,hidden_dim) * 0.01  # Similar to w1 initialization
h_0 = np.zeros((hidden_dim,1)) # First default hidden state 

# 3. Weights from Hidden Layer to Output Layer (Shape: 2 hidden nodes x 3 outputs)
W_ho = np.random.randn(hidden_dim, output_dim) * 0.01 # Similar to w1 initialization
b_o = np.zeros((1, output_dim)) 

In [13]:
# =======================================================
# THE FORWARD PASS (First Time Step: Inputting "a")
# =======================================================

In [23]:
# Prepare time-step 1 column vectors

t = 1

x_1 = X_train[0].reshape(-1, 1)          # Shape (3, 1) - One-hot vector for "a"
h_prev = h_0.reshape(hidden_dim, 1)      # Shape (2, 1) - Initial hidden state

# Step 1: Compute Hidden State using transposes
raw_h = np.dot(W_xh.T, x_1) + np.dot(W_hh.T, h_prev) + b_h.T
h_1 = np.tanh(raw_h)                     # Shape (2, 1)

# Step 2: Compute Unnormalized Output Scores (Logits)
logits = np.dot(W_ho.T, h_1) + b_o.T     # Shape (3, 1)

# Step 3: Compute Softmax Probabilities
exp_logits = np.exp(logits)
y_hat = exp_logits / np.sum(exp_logits)  # Shape (3, 1)

print(f"--- Final Output Probabilities (y_hat) at  t={t} ---")
for word, prob in zip(vocabulary, y_hat.flatten()):
    print(f"  Probability of '{word}': {prob * 100:.2f}%")

--- Final Output Probabilities (y_hat) at  t=1 ---
  Probability of 'a': 33.33%
  Probability of 'b': 33.33%
  Probability of 'c': 33.34%


In [25]:
# Prepare time-step 2 column vectors

t = 2

x_2 = X_train[1].reshape(-1, 1)          
h_prev = h_1.reshape(hidden_dim, 1)     

# Step 1: Compute Hidden State using transposes
raw_h = np.dot(W_xh.T, x_2) + np.dot(W_hh.T, h_prev) + b_h.T
h_2 = np.tanh(raw_h)                  

# Step 2: Compute Unnormalized Output Scores (Logits)
logits = np.dot(W_ho.T, h_2) + b_o.T    

# Step 3: Compute Softmax Probabilities
exp_logits = np.exp(logits)
y_hat = exp_logits / np.sum(exp_logits)  

print(f"--- Final Output Probabilities (y_hat) at t={t} ---")
for word, prob in zip(vocabulary, y_hat.flatten()):
    print(f"  Probability of '{word}': {prob * 100:.2f}%")

--- Final Output Probabilities (y_hat) at t=2 ---
  Probability of 'a': 33.33%
  Probability of 'b': 33.33%
  Probability of 'c': 33.34%


In [27]:
# if WE look closely we have not acheived anything yet , we just got some default outputs without the loss function and applying bptt.
# To actually minimize loss we need to bptt to get actual probabilities